## 06B Baseline Model for Cancellation

#### Purpose
- Develop the baseline model for **lapse** (`LAPSE`: a cancellation during the term or a non-renewal at its end), with the same method as the claim baseline (06a).

#### Steps:
1. Load the features (time split)
2. The target: LAPSE and its parts
3. Naive benchmarks (base rate, new business × mandatory purchase)
4. Preprocessing pipeline
5. Tune with time-based folds, then train logistic regression on every training row
6. Test results (PR-AUC, ROC-AUC, Brier, calibration, lift, confusion matrices)
7. Coefficient interpretation
8. Save the model and results

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III. 

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-16

### Section 0. Load the libraries and working directory
Use the same setup cell as notebook 02 (repo root, data folders, reports folder), so the notebook runs on every teammate's machine.

#### 0.1 Imports, paths and random seed

In [ ]:
%load_ext autoreload
%autoreload 2

# Find the repo root from any folder, so `src` can be imported
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.paths import DOCS, INTERIM, PROCESSED, RAW, REPORTS, ensure_dirs
from src.db import connect, scalar
from src.dq import CleaningLog
from src.features import apply_fixed_rules
from src.modeling import QuantileClipper
pd.options.display.float_format = "{:,.4g}".format     # 4 significant digits, so small rates and alphas stay readable

ensure_dirs()

### Section 1. Load the features and split

#### 1.1 Load the lapse modeling table
We load the lapse population one split at a time: **train** (2014–2020) to fit and **test** (2021–2022) to evaluate. **Validation** (2023 to Aug 2024) stays untouched until notebook 08. The training split is read in chunks, so the 26.8M rows never have to fit in memory at once. Unlike the claim model, we keep the place features (rainfall, ACS, location): for lapse they don't reverse between the training and test years.

In [ ]:
# Step 1.1 Settings and the chunk loader for the lapse population
import time
import numpy as np
from src.features import apply_fixed_rules

SEED = 42
TARGET = "LAPSE"            # a cancellation during the term or a non-renewal at its end (notebook 04)
ID_COLS = ["period_id", "period_year"]
N_CHUNKS = 28              # the 26.8M training rows are read in 28 chunks of about 1M rows

# The features we use, their kind and their missing-value plan, from notebook 05
plan = pd.read_csv(DOCS / "feature_plan.csv")
plan = plan[plan["use"]]

# Unlike the claim model (06a), the place features are kept: for lapse they don't reverse between the training
# and test years, and they help a little (test ROC-AUC 0.656 with them, 0.645 without, in our check).
EXCLUDE = []
plan = plan[~plan["feature"].isin(EXCLUDE)]
FEATS = plan["feature"].tolist()
NUMS = plan.loc[plan["kind"] == "num", "feature"].tolist()
CATS = plan.loc[plan["kind"] == "cat", "feature"].tolist()
FLAGS = plan.loc[plan["kind"] == "flag", "feature"].tolist()
print(f"{len(FEATS)} features: {len(NUMS)} numbers, {len(CATS)} categories, {len(FLAGS)} flags")

con = connect(db_path=None, memory_limit="4GB")
con.execute(f'''CREATE OR REPLACE VIEW lapse_pop AS
                SELECT * FROM read_parquet('{PROCESSED / "features.parquet"}') WHERE in_pop_lapse''')

def prepare(df):
    '''Fixed clean-up rules (notebook 05) and compact types; returns X, y, ids.'''
    df = apply_fixed_rules(df)
    df[NUMS] = df[NUMS].astype("float32")
    df[FLAGS] = df[FLAGS].astype("float32")
    cats = df[CATS].astype("string")
    df[CATS] = cats.astype(object).where(cats.notna(), np.nan)     # plain text, NaN where missing
    return df[FEATS], df[TARGET].astype("int8"), df[ID_COLS]

def load_rows(split, where="TRUE", sample_rows=None, seed=SEED):
    '''Rows of one split. where: extra filter; sample_rows: a random sample of that size (same rows for the same seed).'''
    query = f"SELECT {', '.join(ID_COLS + FEATS)}, {TARGET} FROM lapse_pop WHERE split = '{split}' AND {where}"
    if sample_rows:
        query = f"SELECT * FROM ({query}) USING SAMPLE reservoir({sample_rows} ROWS) REPEATABLE ({seed})"
    return prepare(con.sql(query).df())

def chunk_filter(k, epoch):
    '''Chunk k of N_CHUNKS: a hash of the row id, different for every epoch, so each chunk is a random mix of
    states and years (the file itself is ordered by state, which would mislead the model).'''
    return f"hash(period_id || '{SEED}-{epoch}') % {N_CHUNKS} = {k}"

### Section 2. The target

#### 2.1 `LAPSE` and its parts
`LAPSE` = a cancellation during the term or a non-renewal at its end (notebook 04). Non-renewal is almost all of it; cancellations during the term are 0.1–0.3% of policy-years. We model the combined target and keep the two parts for description.

In [ ]:
# Step 2.1 The target: LAPSE and its two parts, by split
con.sql('''
    SELECT split, MIN(period_year) AS first_year, MAX(period_year) AS last_year, COUNT(*) AS policy_years,
           ROUND(100 * AVG(CAST(LAPSE AS DOUBLE)), 2)          AS lapse_pct,
           ROUND(100 * AVG(CAST(NON_RENEWAL AS DOUBLE)), 2)    AS non_renewal_pct,
           ROUND(100 * AVG(CAST(CANCEL_IN_TERM AS DOUBLE)), 2) AS cancel_in_term_pct
    FROM lapse_pop GROUP BY split ORDER BY first_year
''').df()

### Section 3. Naive benchmark

#### 3.1 Naive benchmarks
Two simple predictions, learned on the training split: the **base rate** (the training lapse rate for every policy-year) and the **segment rate** (the training lapse rate of each new business × mandatory purchase segment, the two strongest simple drivers in Q4 and Q5). We record their PR-AUC, ROC-AUC and Brier score on the **test** split.

In [ ]:
# Step 3.1 Naive benchmarks, learned on the training split only
# (1) base rate: the training lapse rate for every row
# (2) segment rate: the training lapse rate of each new business x mandatory purchase segment (Q4, Q5)
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

base_rate = scalar(con, f"SELECT AVG(CAST({TARGET} AS DOUBLE)) FROM lapse_pop WHERE split = 'train'")
segment_rates = con.sql(f'''
    SELECT CAST(COALESCE(tenure_years >= 0 AND tenure_years < 1, FALSE) AS INTEGER) AS new_business,  -- as apply_fixed_rules
           CAST(COALESCE(mandatory_purchase, FALSE) AS INTEGER)                   AS mandatory_purchase,
           COUNT(*)                                                               AS train_rows,
           AVG(CAST({TARGET} AS DOUBLE))                                           AS segment_rate
    FROM lapse_pop WHERE split = 'train'
    GROUP BY ALL ORDER BY segment_rate DESC
''').df()
print(f"training lapse rate: {base_rate:.4%}")
segment_rates.style.format({"train_rows": "{:,}", "segment_rate": "{:.3%}"})

### Section 4. Preprocessing pipeline

#### 4.1 Build the pipeline
Same structure as 06a: imputation (following the missing-value plan from notebook 05), caps at the 1st and 99th percentile, scaling and one-hot encoding, fitted on a 1M-row sample of the training split only.

In [ ]:
# Step 4.1 Preprocessing, fitted on a 1M-row sample of train (medians, caps, scaling and categories)
import time
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.modeling import QuantileClipper          # in src/, so the saved model can be loaded in other notebooks

# The numbers, grouped by their missing-value plan (6.1 in notebook 05)
mp = plan.set_index("feature")["missing_plan"]
num_fill0 = [f for f in NUMS if mp[f].startswith("fill 0")]                    # first term: 0 and a flag
num_fill99 = [f for f in NUMS if mp[f].startswith("fill 99")]                  # no earlier claim: 99
num_flag = [f for f in NUMS if mp[f].startswith("median (train) + missing flag")]
num_noflag = [f for f in NUMS if f not in num_fill0 + num_fill99 + num_flag]  # median, no flag (incl. the leak cases)

def numbers(imputer):
    return make_pipeline(imputer, QuantileClipper(), StandardScaler())

preprocess = ColumnTransformer([
    ("num_fill0", numbers(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)), num_fill0),
    ("num_fill99", numbers(SimpleImputer(strategy="constant", fill_value=99)), num_fill99),
    ("num_flag", numbers(SimpleImputer(strategy="median", add_indicator=True)), num_flag),
    ("num_noflag", numbers(SimpleImputer(strategy="median")), num_noflag),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="(missing)"),
                          OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=16,
                                        sparse_output=False, dtype=np.float32)), CATS),
    ("flag", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), FLAGS),
], verbose_feature_names_out=False)

t0 = time.time()
X_fit, y_fit, _ = load_rows("train", sample_rows=1_000_000)
preprocess.fit(X_fit)
n_cols = len(preprocess.get_feature_names_out())
print(f"preprocessing fitted on {len(X_fit):,} rows in {time.time() - t0:.0f} s: {len(FEATS)} features -> {n_cols} model columns")

In [ ]:
# Step 4.2 Check: what the model sees (on 5,000 rows of the fitting sample)
Z = preprocess.transform(X_fit.iloc[:5000])
cols = preprocess.get_feature_names_out()
print(f"model columns: {Z.shape[1]}; missing values left after preprocessing: {int(np.isnan(Z).sum())}")

summary = pd.DataFrame(
    [{"group": name, "input features": len(feats),
      "model columns": len(preprocess.named_transformers_[name].get_feature_names_out(feats))}
     for name, _, feats in preprocess.transformers_ if name != "remainder"])
display(summary)

# Numbers should now have mean about 0 and standard deviation about 1; one-hot and flags are 0/1
scaled = [c for c in cols if c in NUMS]
idx = [list(cols).index(c) for c in scaled]
print(f"{len(scaled)} scaled numbers: mean {Z[:, idx].mean():.3f}, average std {Z[:, idx].std(axis=0).mean():.3f}")
del X_fit, y_fit, Z

### Section 5. Logistic regression

#### 5.0 Tune with time-based folds
We choose the penalty (`alpha`) and the step size (`eta0`) with rolling-origin folds: learn on the years before, check on the next year (2017, 2018, 2019, 2020). Tuning uses a 1.5M-row sample of the training split; the test and validation years are not used.

In [ ]:
# Step 5.0 Tune the penalty and step size with time-based folds, on a sample of train
from sklearn.base import clone
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

TUNE_ROWS = 1_500_000      # random rows from 2014-2020 (same rows every run)
X_tune, y_tune, ids_tune = load_rows("train", sample_rows=TUNE_ROWS)

# Rolling-origin folds (notebook 04): train on every year before, check on the next year.
# The first check year is 2017, so every fold has at least 3 years to learn from.
years = ids_tune["period_year"].to_numpy()
folds = [(np.where(years < y)[0], np.where(years == y)[0]) for y in range(2017, 2021)]
for (tr, te), y in zip(folds, range(2017, 2021)):
    print(f"fold checking {y}: learn on {len(tr):,} rows ({years[tr].min()}-{years[tr].max()}), check on {len(te):,}")

pipe = make_pipeline(clone(preprocess),          # preprocessing is refitted inside every fold
                     SGDClassifier(loss="log_loss", learning_rate="invscaling", power_t=0.25,
                                   average=True, max_iter=5, tol=None, random_state=SEED))
param_grid = {
    "sgdclassifier__alpha": [1e-6, 1e-5, 1e-4],  # L2 penalty
    "sgdclassifier__eta0": [0.001, 0.01],        # starting step size
}
search = GridSearchCV(pipe, param_grid, cv=folds,
                      scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc", "log_loss": "neg_log_loss"},
                      refit=False,               # the final model is trained on every row, chunk by chunk (Step 5.1)
                      n_jobs=1, verbose=1)       # one job: parallel jobs fail on Python 3.14 with a class defined in the notebook
t0 = time.time()
search.fit(X_tune, y_tune)
print(f"done in {(time.time() - t0) / 60:.1f} min")

results = pd.DataFrame(search.cv_results_)
cols = ["param_sgdclassifier__alpha", "param_sgdclassifier__eta0", "mean_test_pr_auc", "std_test_pr_auc",
        "mean_test_roc_auc", "mean_test_log_loss"] + [f"split{i}_test_pr_auc" for i in range(len(folds))]
results = results[cols].rename(columns={f"split{i}_test_pr_auc": f"pr_auc_{y}" for i, y in enumerate(range(2017, 2021))})
best = results.sort_values("mean_test_pr_auc", ascending=False).iloc[0]
BEST_ALPHA, BEST_ETA0 = float(best["param_sgdclassifier__alpha"]), float(best["param_sgdclassifier__eta0"])
print(f"best: alpha={BEST_ALPHA}, eta0={BEST_ETA0}")
results.sort_values("mean_test_pr_auc", ascending=False).round(4)

#### 5.1 Fit on every training row
Logistic regression (`SGDClassifier` with log loss) trained on all 26.8M training rows, chunk by chunk, in 3 passes, with the settings from Step 5.0. **No class weights**, so the probabilities stay on the real lapse scale. Tenure, premium change and mandatory purchase are among the features.

In [ ]:
# Step 5.1 Logistic regression on every training row, chunk by chunk (stochastic gradient descent),
# with the penalty and step size chosen in Step 5.0
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

N_EPOCHS = 3               # passes over the full training data (about 4-5 minutes each)

model = SGDClassifier(loss="log_loss",                     # logistic regression
                      alpha=BEST_ALPHA,                    # L2 penalty, from Step 5.0
                      learning_rate="invscaling", eta0=BEST_ETA0, power_t=0.25,   # step size, from Step 5.0
                      average=True,                        # average the coefficients over the steps: steadier result
                      random_state=SEED)

# A fixed 1M-row sample of the test split (2021-2022) to follow the progress after each pass
X_watch, y_watch, _ = load_rows("test", sample_rows=1_000_000)
Z_watch = preprocess.transform(X_watch)

rng = np.random.default_rng(SEED)
history = []
for epoch in range(N_EPOCHS):
    t0, n_seen = time.time(), 0
    for k in rng.permutation(N_CHUNKS):                        # the chunks in a different order every pass
        X, y, _ = load_rows("train", where=chunk_filter(k, epoch))
        order = rng.permutation(len(y))                        # shuffle the rows inside the chunk too
        model.partial_fit(preprocess.transform(X.iloc[order]), y.to_numpy()[order], classes=[0, 1])
        n_seen += len(y)
    p = model.predict_proba(Z_watch)[:, 1]
    history.append({"epoch": epoch + 1, "rows_seen": n_seen, "minutes": round((time.time() - t0) / 60, 1),
                    "log_loss": log_loss(y_watch, p), "roc_auc": roc_auc_score(y_watch, p),
                    "pr_auc": average_precision_score(y_watch, p), "brier": brier_score_loss(y_watch, p)})
    print(history[-1])
del X, y, X_watch, Z_watch
pd.DataFrame(history)

### Section 6. Test results (2021–2022)

#### 6.1 Score the test split
We score every test row (7.1M), chunk by chunk, with the model and the two benchmarks.

In [ ]:
# Step 6.1 Score every test row (2021-2022), chunk by chunk; the validation split stays untouched until notebook 08
t0 = time.time()
parts = []
for k in range(N_CHUNKS):
    X, y, ids = load_rows("test", where=chunk_filter(k, 0))
    parts.append(pd.DataFrame({
        "period_id": ids["period_id"].to_numpy(),
        "period_year": ids["period_year"].to_numpy(),
        "y": y.to_numpy(),
        "p_logreg": model.predict_proba(preprocess.transform(X))[:, 1],
        "new_business": X["new_business"].fillna(0).astype(int).to_numpy(),
        "mandatory_purchase": X["mandatory_purchase"].fillna(0).astype(int).to_numpy(),
    }))
pred = pd.concat(parts, ignore_index=True)
del parts, X, y, ids

# The two benchmarks for the same rows (a segment not seen in training gets the base rate)
pred["p_base"] = base_rate
pred = pred.merge(segment_rates[["new_business", "mandatory_purchase", "segment_rate"]],
                  on=["new_business", "mandatory_purchase"], how="left")
pred["p_segment"] = pred.pop("segment_rate").fillna(base_rate)
print(f"scored {len(pred):,} test rows in {(time.time() - t0) / 60:.1f} min; lapse rate {pred['y'].mean():.3%}")

#### 6.2 Results against the benchmarks
PR-AUC, ROC-AUC, Brier score and log loss, for 2021–2022 together and for each year.

In [ ]:
# Step 6.2 Results: the model against the two benchmarks, for 2021-2022 and for each year
MODELS = {"base rate": "p_base", "segment rate": "p_segment", "logistic regression": "p_logreg"}

def scores(y, p):
    return {"rows": len(y), "claims": int(y.sum()), "claim_rate": y.mean(),
            "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "brier": brier_score_loss(y, p), "log_loss": log_loss(y, p, labels=[0, 1])}

rows = []
for scope, part in [("2021-2022", pred), *[(str(y), g) for y, g in pred.groupby("period_year")]]:
    for name, col in MODELS.items():
        rows.append({"scope": scope, "model": name, **scores(part["y"], part[col])})
results = pd.DataFrame(rows)
results.style.format({"rows": "{:,}", "claims": "{:,}", "claim_rate": "{:.3%}", "roc_auc": "{:.4f}",
                      "pr_auc": "{:.4f}", "brier": "{:.5f}", "log_loss": "{:.4f}"})

#### 6.3 Calibration and lift
**Calibration:** do the predicted probabilities match the actual lapse rates, decile by decile? **Lift:** what share of lapses falls in the riskiest 10%, 20%, … of policy-years?

In [ ]:
# Step 6.3 Calibration and lift by decile of the predicted probability (test, 2021-2022)
pred["decile"] = pd.qcut(pred["p_logreg"].rank(method="first"), 10, labels=range(1, 11)).astype(int)  # 10 = highest risk
deciles = (pred.groupby("decile")
               .agg(rows=("y", "size"), lapses=("y", "sum"), predicted=("p_logreg", "mean"), actual=("y", "mean"))
               .sort_index(ascending=False))
deciles["share_of_lapses"] = deciles["lapses"] / deciles["lapses"].sum()
deciles["cumulative_share_of_lapses"] = deciles["share_of_lapses"].cumsum()
deciles["lift"] = deciles["actual"] / pred["y"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
lim = max(deciles["predicted"].max(), deciles["actual"].max()) * 1.1
ax1.plot([0, lim], [0, lim], color="#9a9a9a", linewidth=1, linestyle="--", label="perfect calibration")
ax1.plot(deciles["predicted"], deciles["actual"], color="#2a6fb0", linewidth=2, marker="o", markersize=6, label="logistic regression")
ax1.set(title="Calibration by decile (test, 2021-2022)", xlabel="mean predicted lapse probability",
        ylabel="actual lapse rate", xlim=(0, lim), ylim=(0, lim))
ax1.legend(frameon=False)
share_policies = np.arange(0, 11) / 10
ax2.plot(share_policies, np.r_[0, deciles["cumulative_share_of_lapses"]], color="#2a6fb0", linewidth=2, marker="o",
         markersize=6, label="logistic regression")
ax2.plot([0, 1], [0, 1], color="#9a9a9a", linewidth=1, linestyle="--", label="random order")
ax2.set(title="Share of lapses captured (test, 2021-2022)", xlabel="share of policy-years, highest predicted risk first",
        ylabel="share of lapses captured", xlim=(0, 1), ylim=(0, 1))
ax2.legend(frameon=False)
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e6e6")
plt.tight_layout()
plt.savefig(REPORTS / "fig_lapse_baseline_calibration_lift.png", dpi=150)
plt.show()
deciles.style.format({"rows": "{:,}", "lapses": "{:,}", "predicted": "{:.3%}", "actual": "{:.3%}",
                      "share_of_lapses": "{:.1%}", "cumulative_share_of_lapses": "{:.1%}", "lift": "{:.2f}"})

#### 6.4 Confusion matrices
A confusion matrix needs a cut-off. We flag the **riskiest 10%, 20% and 30%** of policy-years, using cut-offs learned on the training split, and show the default 0.5 for comparison. Precision, recall and lift at each rule are the business view (for example, which policyholders to contact before renewal); PR-AUC, ROC-AUC and Brier remain our main metrics.

In [ ]:
# Step 6.4 Confusion matrices at alert thresholds chosen on the TRAINING split
# The model gives a probability; a confusion matrix needs a cut-off. With 12% lapses, the usual 0.5 flags very few
# policy-years. We flag the riskiest 10%, 20% and 30% (cut-offs from training predictions, never from test outcomes),
# and show 0.5 for comparison.
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

X_s, _, _ = load_rows("train", sample_rows=1_000_000)
p_train = model.predict_proba(preprocess.transform(X_s))[:, 1]
del X_s

rows = []
for flag_share in [0.10, 0.20, 0.30, None]:
    cut = 0.5 if flag_share is None else float(np.quantile(p_train, 1 - flag_share))
    flagged = (pred["p_logreg"] >= cut).astype(int)
    tn, fp, fn, tp = confusion_matrix(pred["y"], flagged, labels=[0, 1]).ravel()
    rows.append({"rule": "p >= 0.5 (default)" if flag_share is None else f"riskiest {flag_share:.0%} in train",
                 "threshold": cut, "flagged_share": flagged.mean(), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "accuracy": (tp + tn) / len(flagged),
                 "precision": precision_score(pred["y"], flagged, zero_division=0),
                 "recall": recall_score(pred["y"], flagged, zero_division=0),
                 "f1": f1_score(pred["y"], flagged, zero_division=0)})
cm_table = pd.DataFrame(rows)
cm_table["lift"] = cm_table["precision"] / pred["y"].mean()

# The 20% rule as a 2 x 2 table
r = cm_table.iloc[1]
cm20 = pd.DataFrame([[r["TN"], r["FP"]], [r["FN"], r["TP"]]],
                    index=["actual: stayed", "actual: lapsed"], columns=["flagged: no", "flagged: yes"])
display(cm20.style.format("{:,}"))
cm_table.style.format({"threshold": "{:.4f}", "flagged_share": "{:.1%}", "TP": "{:,}", "FP": "{:,}", "FN": "{:,}",
                       "TN": "{:,}", "accuracy": "{:.3f}", "precision": "{:.3%}", "recall": "{:.1%}",
                       "f1": "{:.3f}", "lift": "{:.2f}"})

### Section 7. Coefficient interpretation

#### 7.1 Main coefficients
The strongest effects as odds ratios. We check that they make sense: premium increases and new business should raise lapse, a mortgage requirement and longer tenure should lower it.

In [ ]:
# Step 7.1 Coefficients as odds ratios
# Numbers are standardized, so an odds ratio is the change in the odds of a lapse for +1 standard deviation.
# Categories are one-hot with no level dropped, so compare levels within a feature (e.g. zone V vs zone X).
names = preprocess.get_feature_names_out()

def source_feature(col):
    '''The feature a model column comes from (one-hot level, missing flag or the feature itself).'''
    col = col.removeprefix("missingindicator_")
    return max((f for f in FEATS if col == f or col.startswith(f + "_")), key=len, default=col)

coefs = pd.DataFrame({"model_column": names, "coef": model.coef_.ravel()})
coefs["feature"] = coefs["model_column"].map(source_feature)
coefs["kind"] = np.select([coefs["model_column"].str.startswith("missingindicator_"),
                           coefs["feature"].isin(CATS)], ["missing flag", "category level"], "number / flag")
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs = coefs.sort_values("coef", ascending=False).reset_index(drop=True)
print(f"intercept {model.intercept_[0]:.3f} (base odds {np.exp(model.intercept_[0]):.4f}); {len(coefs)} model columns")

top = pd.concat([coefs.head(15), coefs.tail(15)])
fig, ax = plt.subplots(figsize=(8, 8))
colors = np.where(top["coef"] > 0, "#c0504d", "#2a6fb0")        # red: raises the odds of a lapse; blue: lowers them
ax.barh(top["model_column"][::-1], top["odds_ratio"][::-1] - 1, left=1, color=colors[::-1], height=0.6)
ax.axvline(1, color="#555555", linewidth=1)
ax.set(title="Largest effects: odds ratio of a lapse (logistic regression)",
       xlabel="odds ratio (1 = no effect; red raises the odds of a lapse, blue lowers them)")
ax.set_xscale("log")
lo, hi = top["odds_ratio"].min(), top["odds_ratio"].max()
ticks = [t for t in (0.25, 0.33, 0.5, 0.67, 0.8, 1, 1.25, 1.5, 2, 3, 4) if lo / 1.1 <= t <= hi * 1.1]
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.minorticks_off()
ax.tick_params(axis="y", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(REPORTS / "fig_lapse_baseline_odds_ratios.png", dpi=150)
plt.show()
top[["model_column", "feature", "kind", "coef", "odds_ratio"]].round(3)

### Section 8. Save

#### 8.1 Save the model and results
We save the fitted model to `data/processed/models/`, the test predictions to `data/processed/predictions/`, and the metrics, deciles, confusion matrices, coefficients and run record to `reports/`. The last line reloads the saved model and checks it gives the same predictions.

In [ ]:
# Step 8.1 Save the model, the test predictions and the results
import json
import joblib
from datetime import datetime
from src.modeling import make_portable

MODEL_DIR = PROCESSED / "models"
PRED_DIR = PROCESSED / "predictions"
for d in (MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)
NAME = "lapse_baseline_logreg"

# The model: preprocessing + logistic regression, with any notebook-defined clipper swapped for src.modeling's
print("clippers made portable:", make_portable(preprocess))
joblib.dump({"preprocess": preprocess, "model": model, "features": FEATS}, MODEL_DIR / f"{NAME}.joblib")

# Test predictions, row by row, so notebook 08 can compare models without retraining
pred[["period_id", "period_year", "y", "p_logreg", "p_segment", "p_base"]].to_parquet(
    PRED_DIR / f"{NAME}_test.parquet", index=False)

# Results tables
results.to_csv(REPORTS / f"{NAME}_test_metrics.csv", index=False)
deciles.to_csv(REPORTS / f"{NAME}_test_deciles.csv")
coefs.to_csv(REPORTS / f"{NAME}_coefficients.csv", index=False)
cm_table.to_csv(REPORTS / f"{NAME}_test_confusion.csv", index=False)
segment_rates.to_csv(REPORTS / "lapse_benchmark_segment_rates.csv", index=False)

# Run record: what was trained, on what, with which settings
run = {
    "model": NAME, "target": TARGET, "saved_at": datetime.now().isoformat(timespec="seconds"),
    "train_split": "train (2014-2020), every row, in chunks", "scored_on": "test (2021-2022)",
    "features_used": FEATS, "features_excluded": list(EXCLUDE),
    "model_columns": int(len(preprocess.get_feature_names_out())),
    "params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
               for k, v in model.get_params().items()},
    "epochs": N_EPOCHS, "chunks": N_CHUNKS, "seed": SEED, "base_rate_train": base_rate,
    "test_overall": results.query("scope == '2021-2022' and model == 'logistic regression'").iloc[0].drop(["scope", "model"]).to_dict(),
}
(REPORTS / f"{NAME}_run.json").write_text(json.dumps(run, indent=2, default=float))

# Check: the saved model loads and gives the same predictions
saved = joblib.load(MODEL_DIR / f"{NAME}.joblib")
X_check, _, _ = load_rows("test", sample_rows=1000)
same = np.allclose(saved["model"].predict_proba(saved["preprocess"].transform(X_check))[:, 1],
                   model.predict_proba(preprocess.transform(X_check))[:, 1])
print(f"saved {NAME}.joblib, {NAME}_test.parquet, metrics, deciles, coefficients and run record; reload gives the same predictions: {same}")